# 00 — Setup del validation_study

Verifica paths, importa scripts, lista el grid, valida conectividad con datos del TFG y ejecuta tests del `cleanup_pipeline`.

In [ ]:
# [SETUP] Imports y paths
import sys
from pathlib import Path

study_root = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(study_root))

from scripts import config as cfg
from scripts import grid_utils
from scripts import sample_generation
from scripts import cleanup_pipeline


In [ ]:
# [VERIFY] Paths
print(f"PROJECT_ROOT: {cfg.PROJECT_ROOT}")
print(f"STUDY_ROOT:   {cfg.STUDY_ROOT}")
print(f"TFG_DATA_RAW exists: {cfg.TFG_DATA_RAW.exists()}")
print(f"TFG_DATA_QC exists:  {cfg.TFG_DATA_QC.exists()}")

for path in [cfg.STUDY_SAMPLES, cfg.STUDY_FEATURES, cfg.STUDY_MASTERS,
             cfg.STUDY_SCREENING, cfg.STUDY_FINETUNING, cfg.STUDY_INFORMES,
             cfg.STUDY_INFORMES / "_ARTIFACTS"]:
    path.mkdir(parents=True, exist_ok=True)
print("\n[OK] Carpetas creadas/verificadas")


In [ ]:
# [EXEC] Listar grid
import pandas as pd

grid = grid_utils.generate_grid()
print(f"Total configs: {len(grid)}")
print(f"  Bloque L (long-horizon): {sum(1 for c in grid if c[\'block\']==\'L\')}")
print(f"  Bloque S (short con transac): {sum(1 for c in grid if c[\'block\']==\'S\')}")

grid_df = pd.DataFrame(grid)
print(f"\nBaseline:")
print(grid_df[grid_df['is_baseline']][['id','cutoff','spike','min_logins']].to_string(index=False))
print(f"\nPrimeras 3 configs S:")
print(grid_df[grid_df['block']=='S'].head(3)[['id','cutoff','spike','min_logins']].to_string(index=False))


In [ ]:
# [VERIFY] users.csv del TFG
users_csv = cfg.TFG_DATA_RAW / "users.csv"
assert users_csv.exists(), f"NO ENCONTRADO: {users_csv}"
size_mb = users_csv.stat().st_size / 1024**2
print(f"users.csv: {size_mb:.1f} MB")


In [ ]:
# [VERIFY] Masters y modelos del TFG (para 02zz_rebaseline)
master_base = cfg.TFG_DATA_QC / "master_table_cutoff.parquet"
master_v3 = cfg.TFG_DATA_QC / "master_table_cutoff_v3_aggressive.parquet"
splits = cfg.TFG_DATA_MODELS / "splits_indices_cutoff.parquet"
best_params_dir = cfg.TFG_INFORMES / "fase3_modeling" / "03g_tuning_catboost"
bp14 = best_params_dir / "best_params_churn_14d.json"
bp30 = best_params_dir / "best_params_churn_30d.json"

print(f"master base (115 cols): {'[OK]' if master_base.exists() else '[NO ENCONTRADO]'}")
print(f"master v3 (80 cols):    {'[OK]' if master_v3.exists() else '[NO ENCONTRADO]'}")
print(f"splits cutoff:          {'[OK]' if splits.exists() else '[NO ENCONTRADO]'}")
print(f"best_params 14d:        {'[OK]' if bp14.exists() else '[NO ENCONTRADO]'}")
print(f"best_params 30d:        {'[OK]' if bp30.exists() else '[NO ENCONTRADO]'}")


In [ ]:
# [VERIFY] Tests del cleanup_pipeline
import subprocess
result = subprocess.run(
    ['python', '-m', 'scripts.tests.test_cleanup_pipeline'],
    capture_output=True, text=True, cwd=str(cfg.STUDY_ROOT)
)
print(result.stdout)
if result.returncode != 0:
    print("[ALERT] TESTS DEL CLEANUP FALLARON")
    print("STDERR:", result.stderr)
else:
    print("[OK] Tests del cleanup pasaron")


In [ ]:
# [REPORT] Setup report
from datetime import datetime

report = f"""# Setup report - Validation Study

**Fecha**: {datetime.now()}

## Configuracion

- RANDOM_SEED: {cfg.RANDOM_SEED}
- Total configs en grid: {len(grid)}
- Block L: {sum(1 for c in grid if c['block']=='L')}
- Block S: {sum(1 for c in grid if c['block']=='S')}
- Baseline ID: {grid_df[grid_df['is_baseline']].iloc[0]['id']}

## Cleanup dinamico

- high_nan_threshold: {cfg.CLEANUP_THRESHOLDS['high_nan_threshold']}
- quasi_constant_threshold: {cfg.CLEANUP_THRESHOLDS['quasi_constant_threshold']}
- corr_threshold_v2: {cfg.CLEANUP_THRESHOLDS['corr_threshold_v2']}
- corr_threshold_v3: {cfg.CLEANUP_THRESHOLDS['corr_threshold_v3']}
- TARGET_LEAKAGE: {len(cfg.TARGET_LEAKAGE_COLS)} cols schema-level

## Tests cleanup_pipeline

9/9 tests pasados (verificado en celda anterior).

## Next

Ejecutar 01_generate_grid_samples.ipynb para generar los 54 samples.
"""

(cfg.STUDY_INFORMES / "00_setup_report.md").write_text(report)
print(f'Setup report: {cfg.STUDY_INFORMES / "00_setup_report.md"}')
